# space: breadth vs. depth

section on space costs: breadth and depth (where `lam_time = 0.0`)

deliverables
- pruned models—show that our space costs work through prunning
- investigate behavior—do models cluster?
    - e.g. in terms of confusion matrices, logits
- investigate internal representations
    - particularly differences between shallow-wide vs. deep-narrow models (grad-cam analysis!)
    - visualize the kernels that the models learn?
    - check whether the images that they fail on are different

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import nn_bowl
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import t
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [3]:
TICK_VALS = [0.0, 1.0, 10.0, 100.0, 1000.0, 10000.0]
TICK_TEXT = ["0", "1", "10", "10<sup>2</sup>", "10<sup>3</sup>", "10<sup>4</sup>"]

In [4]:
cfg_base = nn_bowl.utils.load_config("config_breadth_depth.yaml")
WIDTH = cfg_base.plotting.width.full
nn_bowl.utils.set_plotly_template(cfg_base)
figures_dir = nn_bowl.paths.FIGURES_DIR / "breadth_depth"

# ensuring that all evaluation files exist for all combinations of lam_depth, lam_breadth, lam_time and instance
# nn_bowl.utils.ensure_evaluation_files_exist(cfg_base, skip_pruning=False)



In [5]:
# model df
model_df = nn_bowl.utils.get_model_df(cfg_base)

# filter model_df to only include rows where lam_time is 0.0
model_df = model_df[model_df["costs.time.lam"] == 0.0]
model_df["lam_breadth"] = model_df["costs.space.lam_breadth"]
model_df["lam_depth"] = model_df["costs.space.lam_depth"]
model_df["lam_time"] = model_df["costs.time.lam"]
model_df

  3%|▎         | 3/108 [00:02<01:03,  1.64it/s]

Labels shape for cifar10: (10000,)


100%|██████████| 108/108 [00:19<00:00,  5.54it/s]


,model_name,costs.space.lam_breadth,costs.space.lam_depth,costs.time.lam,instance,cost_depth,cost_breadth,cost_time,accuracy,lam_breadth,lam_depth,lam_time
0,lam_breadth=0.0_lam_depth=0.0_lam=0.0_i0,0.0,0.0,0.0,0,0.130335,0.137259,1.221888,0.8759,0.0,0.0,0.0
1,lam_breadth=0.0_lam_depth=1.0_lam=0.0_i0,0.0,1.0,0.0,0,0.015559,0.075773,1.222902,0.8711,0.0,1.0,0.0
2,lam_breadth=0.0_lam_depth=10.0_lam=0.0_i0,0.0,10.0,0.0,0,0.002551,0.047244,1.223414,0.8674,0.0,10.0,0.0
3,lam_breadth=0.0_lam_depth=100.0_lam=0.0_i0,0.0,100.0,0.0,0,0.000422,0.027157,1.222800,0.8355,0.0,100.0,0.0
4,lam_breadth=0.0_lam_depth=1000.0_lam=0.0_i0,0.0,1000.0,0.0,0,0.000061,0.010933,1.220825,0.7688,0.0,1000.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
103,lam_breadth=10000.0_lam_depth=1.0_lam=0.0_i2,10000.0,1.0,0.0,2,0.002996,0.000003,1.222549,0.7158,10000.0,1.0,0.0
104,lam_breadth=10000.0_lam_depth=10.0_lam=0.0_i2,10000.0,10.0,0.0,2,0.000669,0.000003,1.220038,0.6906,10000.0,10.0,0.0
105,lam_breadth=10000.0_lam_depth=100.0_lam=0.0_i2,10000.0,100.0,0.0,2,0.000184,0.000003,1.216561,0.6630,10000.0,100.0,0.0
106,lam_breadth=10000.0_lam_depth=1000.0_lam=0.0_i2,10000.0,1000.0,0.0,2,0.000031,0.000002,1.216673,0.6172,10000.0,1000.0,0.0


In [6]:
# lambdas vs actual raw costs in the loss
# (showing that the costs are working)
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.15)
fig.update_yaxes(type="log", exponentformat="power")
fig.update_layout(height=WIDTH * 0.2, width=WIDTH * 0.35)
fig.update_layout(margin=dict(l=45, r=10, t=20, b=60))
# fig.update_layout(margin=dict(l=55, r=10, t=20, b=60))

for i, cost_name in enumerate(["breadth", "depth"]):
    x_col = f"lam_{cost_name}"
    y_col = f"cost_{cost_name}"

    instance_means = model_df.groupby([x_col, "instance"])[y_col].mean().reset_index()
    grouped = instance_means.groupby(x_col)[y_col].apply(
        lambda x: pd.Series({
            "log_mean": np.mean(np.log10(x)),
            "log_std": np.std(np.log10(x), ddof=1),
        })
    ).unstack().reset_index()

    multiplier = t.ppf(0.975, df=cfg_base.experiment.n_instances - 1) 
    log_sem = multiplier * grouped["log_std"] / np.sqrt(cfg_base.experiment.n_instances)

    y_vals = 10 ** grouped["log_mean"]
    y_upper = 10 ** (grouped["log_mean"] + log_sem) - y_vals
    y_lower = y_vals - 10 ** (grouped["log_mean"] - log_sem)

    fig.add_trace(
        go.Scatter(
            x=grouped[x_col],
            y=y_vals,
            error_y=dict(
                type="data",
                symmetric=False,
                array=y_upper,
                arrayminus=y_lower,
                visible=True,
            ),
            mode="lines+markers",
            name=cost_name,
            marker_color=cfg_base.plotting[cost_name],
        ),
        row=1, col=i+1
    )

    fig.update_xaxes(title_text=f"λ<sub>{cost_name}</sub>", row=1, col=i+1)
    fig.update_yaxes(title_text=f"ℒ<sub>{cost_name}</sub>", row=1, col=i+1)

fig.update_xaxes(
    type="category",
    categoryorder="array",
    categoryarray=TICK_VALS,
    tickmode="array",
    tickvals=TICK_VALS,
    ticktext=TICK_TEXT,
    tickangle=0,
    title_standoff=6,
)

fig.update_yaxes(zeroline=False)

# 
tick_text_costs = ["10<sup>-5</sup>", "10<sup>-3</sup>", "10<sup>-1</sup>"]
# tick_vals_costs = [1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1]
tick_vals_costs = [1e-5, 1e-3, 1e-1]
fig.update_yaxes(
    type="log",
    exponentformat="power",
    range=[-6, -1],
    tickmode="array",
    tickvals=tick_vals_costs,
    ticktext=tick_text_costs,
    title_standoff=0,
)
fig.show()

nn_bowl.utils.save_figure(fig, figures_dir, "breadth_depth_lambdas_costs_ci", scale=2)

In [7]:
# lambdas vs accuracy
# accuracy grid plot

# pivot data into a grid
pivot = model_df.pivot_table(
    index="lam_depth",
    columns="lam_breadth",
    values="accuracy",
    aggfunc="mean"
)

fig = go.Figure(data=go.Heatmap(
    z=pivot.values,
    x=[str(int(c)) for c in pivot.columns],
    y=[str(int(r)) for r in pivot.index],
    # colorscale="gray",
    colorscale="Plasma",
    colorbar_title="accuracy",
    xgap=0,
    ygap=0,
    colorbar=dict(orientation="v", len=1.0, thickness=12, outlinewidth=0),
    text=pivot.values.round(2),       # show values in cells
    texttemplate="%{text}",
))



fig.update_xaxes(
    type="category", 
    categoryorder="array", 
    categoryarray=TICK_VALS,
    tickmode="array",
    tickvals=TICK_VALS,
    ticktext=TICK_TEXT,
    # tickangle=0               # Forces labels to be horizontal
)
fig.update_yaxes(
    type="category", 
    categoryorder="array", 
    categoryarray=TICK_VALS,
    tickmode="array",
    tickvals=TICK_VALS,
    ticktext=TICK_TEXT,
    tickangle=0               # Forces labels to be horizontal
)


fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_layout(
    xaxis_title="λ<sub>breadth</sub>",
    yaxis_title="λ<sub>depth</sub>",
    height=WIDTH * 0.25,
    width=WIDTH * 0.35
)
fig.update_xaxes(title_standoff=10)


fig.update_layout(margin=dict(l=70, r=0, t=20, b=70))
fig.show()

nn_bowl.utils.save_figure(fig, figures_dir, "breadth_depth_accuracy_grid", scale=2)


In [8]:

# pruning results and masks
pruning_results_dfs, pruning_masks = nn_bowl.utils.get_pruning_results_and_masks(cfg_base, model_df)

# Load data needed for this cell
all_layer_channel_norms = nn_bowl.utils.get_layer_channel_norms(cfg_base)

100%|██████████| 108/108 [00:01<00:00, 82.83it/s]


In [9]:
CELL_SIZE = (150, 150)
EPS = 1e-12
CMAP = "Viridis"

lam_time = 0.0
instance = 0

tick_text_dict = {0.0: "0", 1.0: "1", 10.0: "10", 100.0: "10<sup>2</sup>", 1000.0: "10<sup>3</sup>", 10000.0: "10<sup>4</sup>"}

lam_depths = np.unique(model_df["costs.space.lam_depth"].values)
lam_breadths = np.unique(model_df["costs.space.lam_breadth"].values)
lam_times = np.unique(model_df["costs.time.lam"].values)

# --- Compute global log-space min/max across all arrays ---
all_values = []
for lam_depth in lam_depths:
    for lam_breadth in lam_breadths:
        combo = {
            "costs.space.lam_breadth": lam_breadth,
            "costs.space.lam_depth": lam_depth,
            "costs.time.lam": lam_time,
            "instance": instance,
        }
        key = nn_bowl.utils.get_combo_key(combo)
        arr = all_layer_channel_norms.get(key, None)
        if arr is not None:
            all_values.append(np.asarray(arr).ravel())

all_values = np.concatenate(all_values)
all_log = np.log10(all_values + EPS)
ZMIN = float(np.nanmin(all_log))
ZMAX = float(np.nanmax(all_log))

# --- Build colorbar ticks in original (linear) units ---
candidate_ticks = np.array([1e-8, 1e-7, 1e-6, 1e-5, 1e-4, 1e-3, 1e-2, 1e-1, 1.0, 10.0, 100.0, 1000.0, 10000.0])
tick_vals_orig = candidate_ticks[(candidate_ticks >= 10**ZMIN) & (candidate_ticks <= 10**ZMAX)]
tick_vals_log = np.log10(tick_vals_orig)
colorbar_tick_text = [f"10<sup>{int(np.log10(v))}</sup>" for v in tick_vals_orig]

# --- Build figure (no x_title/y_title — we'll use annotations) ---
fig = make_subplots(
    rows=len(lam_depths), cols=len(lam_breadths),
    horizontal_spacing=0.02, vertical_spacing=0.02,
)

colorbar_added = False
all_shapes = []

for i, lam_depth in enumerate(reversed(lam_depths), start=1):
    for j, lam_breadth in enumerate(lam_breadths, start=1):

        combo = {
            "costs.space.lam_breadth": lam_breadth,
            "costs.space.lam_depth": lam_depth,
            "costs.time.lam": lam_time,
            "instance": instance,
        }
        key = nn_bowl.utils.get_combo_key(combo)
        arr = all_layer_channel_norms.get(key, None)

        if arr is None:
            fig.add_trace(go.Heatmap(
                z=[[np.nan]], colorscale='viridis',
                zmin=ZMIN, zmax=ZMAX, showscale=False
            ), row=i, col=j)
            continue

        arr_log = np.log10(np.asarray(arr) + EPS)

        # Determine text color based on average brightness of top region
        top_row_mean = arr_log[-1, :].mean()
        mid_val = (ZMIN + ZMAX) / 4
        text_color = "black"  # always dark; the white box below keeps it readable on any background

        show_colorbar = not colorbar_added
        fig.add_trace(go.Heatmap(
            z=arr_log,
            colorscale=CMAP,
            zmin=ZMIN,
            zmax=ZMAX,
            showscale=show_colorbar,
            colorbar=dict(
                title=dict(text="avg. weight magnitude",
                           font=dict(size=cfg_base.plotting.font.size),
                           side="top"),
                len=0.55,
                thickness=15,
                y=1.02,
                x=0.7,
                outlinewidth=0,
                tickvals=tick_vals_log,
                ticktext=colorbar_tick_text,
                orientation="h",
            ) if show_colorbar else None,
            hovertemplate=(
                "layer=%{y}<br>"
                "channel=%{x}<br>"
                "log10(norm)=%{z:.3f}"
                "<extra></extra>"
            )
        ), row=i, col=j)
        if show_colorbar:
            colorbar_added = True

        fig.update_xaxes(showticklabels=True, title_text="",
                         row=i, col=j, autorange=True)
        fig.update_yaxes(showticklabels=True, title_text="",
                         row=i, col=j, autorange=True)

        subplot_idx = (i - 1) * len(lam_breadths) + j
        xref_domain = "x domain" if subplot_idx == 1 else f"x{subplot_idx} domain"
        yref_domain = "y domain" if subplot_idx == 1 else f"y{subplot_idx} domain"
        xref_data = "x" if subplot_idx == 1 else f"x{subplot_idx}"
        yref_data = "y" if subplot_idx == 1 else f"y{subplot_idx}"

        # Y-axis lambda labels (left side)
        if j == 1:
            fig.add_annotation(
                x=-0.8, y=0.5,
                xref=xref_domain, yref=yref_domain,
                text=tick_text_dict[lam_depth],
                showarrow=False,
                font=dict(size=cfg_base.plotting.font.size),
            )

        # X-axis lambda labels (bottom)
        if i == len(lam_depths):
            fig.add_annotation(
                x=0.5, y=-0.95,
                xref=xref_domain, yref=yref_domain,
                text=tick_text_dict[lam_breadth],
                showarrow=False,
                font=dict(size=cfg_base.plotting.font.size),
            )

        pruning_df = pruning_results_dfs.get(key)
        baseline_accuracy = pruning_df[pruning_df["phase"] == "baseline"]["accuracy"].values[0]
        final_accuracy = pruning_df[pruning_df["threshold_hit"] == False]["accuracy"].values[-1]

        # Accuracy annotation inside each subplot
        fig.add_annotation(
            x=0.98, y=0.98,
            xref=xref_domain, yref=yref_domain,
            text=f"{baseline_accuracy:.0%}→{final_accuracy:.0%}",
            showarrow=False,
            font=dict(size=12, color=text_color),
            bgcolor="rgba(255,255,255,0.8)", borderpad=1,
            xanchor="right", yanchor="top",
        )

        mask_arr = pruning_masks.get(key)
        if mask_arr is not None:
            boundary_shapes = nn_bowl.utils.mask_boundary_shapes(mask_arr, color="red", width=2)
            for s in boundary_shapes:
                s["xref"] = xref_data
                s["yref"] = yref_data
                all_shapes.append(s)

fig.update_layout(shapes=all_shapes)

for i, lam_depth in enumerate(reversed(lam_depths), start=1):
    for j, lam_breadth in enumerate(lam_breadths, start=1):
        is_bottom_row = (i == len(lam_depths))
        is_left_col = (j == 1)

        fig.update_xaxes(
            showticklabels=is_bottom_row,
            ticks="outside" if is_bottom_row else "",
            tickvals=[0, 15, 31, 47],
            ticktext=["1", "16", "32", "48"],
            tickfont_size=10,
            row=i, col=j,
        )
        fig.update_yaxes(
            showticklabels=is_left_col,
            ticks="outside" if is_left_col else "",
            tickvals=[0, 1, 2, 3, 4, 5],
            ticktext=["1", "2", "3", "4", "5", "6"],
            tickfont_size=10,
            row=i, col=j,
        )

        if is_bottom_row and is_left_col:
            subplot_idx = (i - 1) * len(lam_breadths) + j
            xref_domain = "x domain" if subplot_idx == 1 else f"x{subplot_idx} domain"
            yref_domain = "y domain" if subplot_idx == 1 else f"y{subplot_idx} domain"

            fig.add_annotation(
                x=0.5, y=-0.55, xref=xref_domain, yref=yref_domain,
                text="channel", showarrow=False, font=dict(size=12),
            )
            fig.add_annotation(
                x=-0.45, y=0.5, xref=xref_domain, yref=yref_domain,
                text="layer", showarrow=False, font=dict(size=12), textangle=-90,
            )

XLAB_Y = -0.2
YLAB_X = -0.17

fig.add_annotation(
    x=0.5, y=XLAB_Y,
    xref="paper", yref="paper",
    text="λ<sub>breadth</sub> →",
    showarrow=False,
    font=dict(size=cfg_base.plotting.font.size),
)
fig.add_annotation(
    x=YLAB_X, y=0.5,
    xref="paper", yref="paper",
    text="λ<sub>depth</sub> →",
    showarrow=False,
    font=dict(size=cfg_base.plotting.font.size),
    textangle=-90,
)

pruned_model_y = 1.14
fig.add_annotation(
    x=0.0, y=pruned_model_y,
    xref="paper", yref="paper",
    text="pruned model",
    showarrow=False,
    font=dict(size=cfg_base.plotting.font.size),
    align="left",
    xanchor="left",
)
fig.add_annotation(
    x=0.0, y=pruned_model_y - 0.05,
    xref="paper", yref="paper",
    text="98% above-chance accuracy preserved",
    showarrow=False,
    font=dict(size=10),
    align="left",
    xanchor="left",
)
fig.add_shape(
    type="rect",
    x0=0.22, x1=0.28, y0=pruned_model_y-0.0, y1=pruned_model_y-0.035,
    xref="paper", yref="paper",
    line=dict(color="red", width=3),
    fillcolor="rgba(0,0,0,0)",
)

fig.update_layout(margin=dict(l=100, r=5, t=0, b=100))
fig.update_layout(height=WIDTH * 0.7, width=WIDTH * 0.65)
fig.show()

nn_bowl.utils.save_figure(fig, figures_dir, "breadth_depth_layer_channel_norms_grid", scale=4)

In [10]:
def add_layers_channels_used(model_df_row):
    lam_breadth = model_df_row["lam_breadth"]
    lam_depth = model_df_row["lam_depth"]
    lam_time = model_df_row["lam_time"]
    instance = model_df_row["instance"]

    combo = {
        "costs.space.lam_breadth": lam_breadth,
        "costs.space.lam_depth": lam_depth,
        "costs.time.lam": lam_time,
        "instance": instance,
    }
    key = nn_bowl.utils.get_combo_key(combo)
    mask = pruning_masks.get(key)
    layers_used = np.sum(np.sum(mask, axis=1) > 0)  # count how many layers have at least one channel used
    channels_used = np.sum(mask) / layers_used

    model_df_row["layers_used"] = layers_used
    model_df_row["channels_used"] = channels_used

    return model_df_row

model_df = model_df.apply(add_layers_channels_used, axis=1)

In [11]:
# lambdas vs resources used according to pruning
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.15)
fig.update_layout(margin=dict(l=45, r=10, t=20, b=60))
fig.update_layout(height=WIDTH * 0.2, width=WIDTH * 0.35)

for i, cost_name in enumerate(["breadth", "depth"]):
    y_col = "channels_used" if cost_name == "breadth" else "layers_used"
    x_col = f"lam_{cost_name}"
    instance_means = model_df.groupby([x_col, "instance"])[y_col].mean().reset_index()
    grouped = instance_means.groupby(x_col)[y_col].apply(
        lambda x: pd.Series({
            "mean": np.mean(x),
            "std": np.std(x, ddof=1),
        })
    ).unstack().reset_index()

    multiplier = t.ppf(0.975, df=cfg_base.experiment.n_instances - 1)
    sem = multiplier * grouped["std"] / np.sqrt(cfg_base.experiment.n_instances)

    fig.add_trace(
        go.Scatter(
            x=grouped[x_col],
            y=grouped["mean"],
            error_y=dict(type="data", array=sem, visible=True),
            mode="lines+markers",
            name=cost_name,
            marker_color=cfg_base.plotting[cost_name],
        ),
        row=1, col=i+1
    )
    fig.update_xaxes(title_text=f"λ<sub>{cost_name}</sub>", row=1, col=i+1)
    title_text = "channels used" if cost_name == "breadth" else "layers used"
    fig.update_yaxes(title_text=title_text, row=1, col=i+1)

fig.update_xaxes(
    type="category",
    categoryorder="array",
    categoryarray=TICK_VALS,
    tickmode="array",
    tickvals=TICK_VALS,
    ticktext=TICK_TEXT,
    tickangle=0,
    title_standoff=6,
)
fig.update_yaxes(zeroline=False, title_standoff=0)
fig.show()
nn_bowl.utils.save_figure(fig, figures_dir, "breadth_depth_resources_used_ci", scale=2)

In [12]:
heatmaps = nn_bowl.utils.load_heatmaps(cfg_base, model_df)

100%|██████████| 108/108 [00:31<00:00,  3.48it/s]


In [13]:
test_dataset = nn_bowl.utils.get_dataloaders(cfg_base)["test"].dataset

In [14]:
heatmaps[0].keys()

dict_keys([35, 39, 107, 129, 251, 321, 439, 576, 582, 799, 952, 971, 1056, 1188, 1315, 1513, 1731, 1760, 1963, 2020, 2249, 2304, 2545, 2592, 2678, 2750, 2753, 2876, 2894, 2927, 3006, 3032, 3039, 3145, 3154, 3305, 3337, 3465, 3570, 3723, 3787, 3857, 3931, 3973, 3999, 4521, 4625, 4640, 4673, 4684, 4742, 4819, 4947, 4949, 4993, 5202, 5272, 5323, 5359, 5547, 5589, 5653, 5861, 6033, 6077, 6231, 6252, 6340, 6363, 6590, 6630, 6687, 7051, 7231, 7438, 7487, 7539, 7770, 7825, 7938, 7942, 7966, 8127, 8158, 8178, 8187, 8243, 8284, 8328, 8362, 8575, 8847, 9001, 9054, 9189, 9238, 9317, 9485, 9655, 9753, 9896, 9920, 9930])

In [20]:
cifar10_indices = [3931, 4949]
# cifar10_indices = list(heatmaps[0].keys())[36:38]
# attribution_methods = ["grad_cam", "image_perturbation"]
attribution_methods = ["image_perturbation"]

# --- layout in pixels (grids are exact squares; everything else is placed around them) ---
FIG_W = WIDTH * 0.7
MARGIN = dict(l=5, r=75, t=8, b=68)          # r: room for the colorbar, b: x tick labels + title (incl. subscript descender)
IMG_FRAC = 0.45                              # original image size relative to the grid
IMG_GAP_PX = 6                               # gap between original image and the y-axis labels
YAXIS_PX = [58, 38]                          # room for y tick labels (+ λ_depth title on the first grid only)
PANEL_GAP_PX = 18                            # gap between the two image+grid panels

plot_w = FIG_W - MARGIN["l"] - MARGIN["r"]
grid_px = (plot_w - sum(YAXIS_PX) - len(cifar10_indices) * IMG_GAP_PX - PANEL_GAP_PX) / (len(cifar10_indices) * (1 + IMG_FRAC))
img_px = IMG_FRAC * grid_px
plot_h = grid_px
FIG_H = plot_h + MARGIN["t"] + MARGIN["b"]

fig = make_subplots(rows=len(attribution_methods), cols=len(cifar10_indices))

x_px = 0.0
for row_idx, attribution_method in enumerate(attribution_methods):
    for col, cifar10_idx in enumerate(cifar10_indices):
        heatmaps_dict = {}
        for ld in lam_depths:
            for lb in lam_breadths:

                model_row = model_df[
                    (model_df["lam_depth"] == ld) &
                    (model_df["lam_breadth"] == lb) &
                    (model_df["instance"] == 0)
                ]
                heatmaps_idx = model_row["heatmaps_idx"].values[0]
                heatmap_model = heatmaps[heatmaps_idx][cifar10_idx][attribution_method]

                if attribution_method in ("grad_cam", "ablation_cam"):  # image_perturbation doesn't depend on layer
                    layers_used = model_row["layers_used"].values[0]
                    heatmap_model = heatmap_model[layers_used - 1]  # last layer used by the model

                heatmaps_dict[(ld, lb)] = nn_bowl.utils.normalize_heatmap(heatmap_model) # to sum to 1

        img_tensor = nn_bowl.utils.denormalize_cifar10(test_dataset[cifar10_idx]["img"])
        img_np = img_tensor.permute(1, 2, 0).cpu().numpy()

        fig.add_trace(
            go.Heatmap(
                z=pivot.values,
                x=[str(int(c)) for c in pivot.columns],
                y=[str(int(r)) for r in pivot.index],
                colorscale=[[0, "rgba(0,0,0,0)"], [1, "rgba(0,0,0,0)"]],
                showscale=False,
                xgap=1, ygap=1,
            ),
            row=row_idx + 1, col=col + 1,
        )

        ax_idx = row_idx * len(cifar10_indices) + col + 1
        xref = "x" if ax_idx == 1 else f"x{ax_idx}"
        yref = "y" if ax_idx == 1 else f"y{ax_idx}"

        for i, ld in enumerate(pivot.index):
            for j, lb in enumerate(pivot.columns):
                fig.add_layout_image(
                    source=nn_bowl.utils.heatmap_overlay_to_base64(img_np, heatmaps_dict[(ld, lb)], colormap="turbo"),
                    x=j, y=i,
                    xref=xref, yref=yref,
                    sizex=0.95, sizey=0.95,
                    xanchor="center", yanchor="middle",
                    layer="above",
                )

        # original image, then y-axis labels, then the square grid
        fig.add_layout_image(
            source=nn_bowl.utils.img_to_base64(img_np),
            x=x_px / plot_w, y=0.5,
            xref="paper", yref="paper",
            sizex=img_px / plot_w, sizey=img_px / plot_h,
            xanchor="left", yanchor="middle",
            layer="above",
        )
        x_px += img_px + IMG_GAP_PX + YAXIS_PX[col]
        grid_domain = [x_px / plot_w, (x_px + grid_px) / plot_w]
        x_px += grid_px + PANEL_GAP_PX

        fig.update_xaxes(
            domain=grid_domain,
            type="category", categoryorder="array", categoryarray=TICK_VALS,
            tickmode="array", tickvals=TICK_VALS, ticktext=TICK_TEXT,
            title_text="λ<sub>breadth</sub>" if row_idx == len(attribution_methods) - 1 else None,
            row=row_idx + 1, col=col + 1,
            title_standoff=5,
        )
        fig.update_yaxes(
            domain=[0, 1],
            type="category", categoryorder="array", categoryarray=TICK_VALS,
            tickmode="array", tickvals=TICK_VALS, ticktext=TICK_TEXT,
            title_text="λ<sub>depth</sub>" if col == 0 else None,
            title_standoff=2,
            row=row_idx + 1, col=col + 1,
        )

# colorbar: heatmaps are min-max scaled per panel (heatmap_overlay_to_base64 without vmin/vmax),
# so the scale is relative (least -> most important pixels within each panel), not comparable across panels
turbo = plt.get_cmap("turbo")
turbo_colorscale = [[v, "rgb({},{},{})".format(*(int(255 * c) for c in turbo(v)[:3]))] for v in np.linspace(0, 1, 11)]
fig.add_trace(go.Scatter(
    x=[None], y=[None], mode="markers", showlegend=False, hoverinfo="skip",
    marker=dict(
        colorscale=turbo_colorscale, cmin=0, cmax=1, color=[0], showscale=True,
        colorbar=dict(
            title=dict(text="attribution", side="right", font=dict(size=12)),
            tickvals=[0, 1], ticktext=["low", "high"], tickfont=dict(size=11),
            thickness=10, len=1.0, y=0.5, yanchor="middle",
            x=1.0, xanchor="left", xpad=8, outlinewidth=0,
        ),
    ),
))

fig.update_layout(height=FIG_H, width=FIG_W, margin=MARGIN)
fig.show()

nn_bowl.utils.save_figure(fig, figures_dir, "breadth_depth_attribution_heatmaps", scale=2)

In [16]:
# quantitative analysis of attribution heatmaps
records = []
for _, row in model_df.iterrows():
    heatmaps_model = heatmaps[row["heatmaps_idx"]]
    for img_idx, methods in heatmaps_model.items():
        hm = methods["image_perturbation"]
        flat = hm.flatten()
        records.append({
            "lam_depth": row["lam_depth"],
            "lam_breadth": row["lam_breadth"],
            "instance": row["instance"],
            "img_idx": img_idx,
            "entropy": np.sum(-flat * np.log(flat + 1e-12))  # add small constant to avoid log(0)
        })

entropy_df = pd.DataFrame(records)
entropy_pivot = entropy_df.groupby(["lam_depth", "lam_breadth"])["entropy"].mean().reset_index()
entropy_grid = entropy_pivot.pivot(index="lam_depth", columns="lam_breadth", values="entropy")



In [17]:
model_df

,model_name,costs.space.lam_breadth,costs.space.lam_depth,costs.time.lam,instance,cost_depth,cost_breadth,cost_time,accuracy,lam_breadth,lam_depth,lam_time,layers_used,channels_used,heatmaps_idx
0,lam_breadth=0.0_lam_depth=0.0_lam=0.0_i0,0.0,0.0,0.0,0,0.130335,0.137259,1.221888,0.8759,0.0,0.0,0.0,6,47.166667,0
1,lam_breadth=0.0_lam_depth=1.0_lam=0.0_i0,0.0,1.0,0.0,0,0.015559,0.075773,1.222902,0.8711,0.0,1.0,0.0,5,40.400000,1
2,lam_breadth=0.0_lam_depth=10.0_lam=0.0_i0,0.0,10.0,0.0,0,0.002551,0.047244,1.223414,0.8674,0.0,10.0,0.0,4,36.000000,2
3,lam_breadth=0.0_lam_depth=100.0_lam=0.0_i0,0.0,100.0,0.0,0,0.000422,0.027157,1.222800,0.8355,0.0,100.0,0.0,3,34.000000,3
4,lam_breadth=0.0_lam_depth=1000.0_lam=0.0_i0,0.0,1000.0,0.0,0,0.000061,0.010933,1.220825,0.7688,0.0,1000.0,0.0,2,32.000000,4
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
103,lam_breadth=10000.0_lam_depth=1.0_lam=0.0_i2,10000.0,1.0,0.0,2,0.002996,0.000003,1.222549,0.7158,10000.0,1.0,0.0,6,8.000000,103
104,lam_breadth=10000.0_lam_depth=10.0_lam=0.0_i2,10000.0,10.0,0.0,2,0.000669,0.000003,1.220038,0.6906,10000.0,10.0,0.0,5,7.200000,104
105,lam_breadth=10000.0_lam_depth=100.0_lam=0.0_i2,10000.0,100.0,0.0,2,0.000184,0.000003,1.216561,0.6630,10000.0,100.0,0.0,4,7.500000,105
106,lam_breadth=10000.0_lam_depth=1000.0_lam=0.0_i2,10000.0,1000.0,0.0,2,0.000031,0.000002,1.216673,0.6172,10000.0,1000.0,0.0,3,6.666667,106


In [18]:
fig = go.Figure()

fig.add_trace(go.Heatmap(
    z=entropy_grid.values,
    x=[str(int(c)) for c in entropy_grid.columns],
    y=[str(int(r)) for r in entropy_grid.index],
    colorscale="Cividis",
    showscale=True,

    colorbar=dict(
        title=dict(text="attribution<br>map<br>entropy", font=dict(size=16), side="top"),
        orientation="v", len=1.0, thickness=12, outlinewidth=0,
        y=0.5, x=1.01, xanchor="left",
        tickfont=dict(size=12),
    ),
    text=entropy_grid.values,
    texttemplate="%{text:.2f}",

))

fig.update_xaxes(
    type="category", categoryorder="array", categoryarray=TICK_VALS,
    tickmode="array", tickvals=TICK_VALS, ticktext=TICK_TEXT,
    title_text="λ<sub>breadth</sub>",
)
fig.update_yaxes(
    type="category", categoryorder="array", categoryarray=TICK_VALS,
    tickmode="array", tickvals=TICK_VALS, ticktext=TICK_TEXT,
    title_text="λ<sub>depth</sub>",
)
fig.update_layout(
    height=WIDTH * 0.25,
    width=WIDTH * 0.3,
    margin=dict(l=50, r=0, t=30, b=60),
)
fig.show()
nn_bowl.utils.save_figure(fig, figures_dir, "breadth_depth_attribution_entropy", scale=2)

In [19]:
# Merge entropy and accuracy data
# Assuming entropy_grid and pivot (accuracy) have the same shape and indices
entropy_flat = entropy_grid.stack().reset_index()
entropy_flat.columns = ["lam_depth", "lam_breadth", "entropy"]

accuracy_flat = pivot.stack().reset_index()
accuracy_flat.columns = ["lam_depth", "lam_breadth", "accuracy"]

# Merge layers used
layers_flat = model_df.pivot_table(
    index="lam_depth", columns="lam_breadth", values="layers_used", aggfunc="mean"
).stack().reset_index()
layers_flat.columns = ["lam_depth", "lam_breadth", "layers_used"]

merged = entropy_flat.merge(accuracy_flat, on=["lam_depth", "lam_breadth"])
merged = merged.merge(layers_flat, on=["lam_depth", "lam_breadth"])

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=merged["accuracy"],
    y=merged["entropy"],
    mode="markers",
    marker=dict(
        size=10,
        color=merged["layers_used"],
        colorscale="Viridis",
        # pad the top of the color range so the "6" tick label clears the colorbar title
        cmin=merged["layers_used"].min(), cmax=merged["layers_used"].max() + 0.35,
        colorbar=dict(title="layers<br>used", thickness=12, len=0.8),
        showscale=True,
        line=dict(width=0.5, color="white"),
    ),
    text=[f"λb={int(r.lam_breadth)}, λd={int(r.lam_depth)}<br>layers={r.layers_used:.1f}" 
          for _, r in merged.iterrows()],
    hovertemplate="%{text}<br>acc=%{x:.2f}<br>entropy=%{y:.2f}<extra></extra>",
))

fig.update_layout(
    xaxis_title="accuracy",
    yaxis_title="attribution map entropy",
    width=WIDTH * 0.3,
    height=WIDTH * 0.25,
    margin=dict(l=60, r=10, t=20, b=60),
)
fig.show()
nn_bowl.utils.save_figure(fig, figures_dir, "entropy_vs_accuracy_by_depth", scale=2)